# Scraping

In [36]:
# pip install pyarrow

In [37]:
"""
TMDB Top Rated Movies Scraper
Fetches movie details using the official TMDB API.

Get your free API key at: https://www.themoviedb.org/settings/api
"""

import requests
import pandas as pd
import time

In [38]:
# CONFIG
API_KEY    = "a7eaa982f861eda59593b7ebf834464c"
PAGES      = 500
BASE_URL   = "https://api.themoviedb.org/3"

API key didapatkan dari website TMDB.

In [39]:
def get_top_rated_movies(page):
    """Fetch one page of top-rated movies (20 per page)."""
    r = requests.get(
        f"{BASE_URL}/movie/top_rated",
        params={"api_key": API_KEY, "language": "en-US", "page": page},
        timeout=10,
    )
    r.raise_for_status()
    return r.json().get("results", [])

Data diambil dari bagian top rated movies. Di ambil 500 pages di mana 1 page berisi 20 film sehingga total film yang diharapkan berjumlah 10000.

In [40]:
def get_movie_details(movie_id):
    """Fetch full details + credits for a single movie."""
    # append_to_response fetches credits in the same request — saves API calls
    r = requests.get(
        f"{BASE_URL}/movie/{movie_id}",
        params={
            "api_key": API_KEY,
            "language": "en-US",
            "append_to_response": "credits,release_dates",
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()

In [41]:
def parse_certificate(release_dates):
    """Extract US content rating (e.g. PG-13) from release_dates response."""
    for country in release_dates.get("results", []):
        if country.get("iso_3166_1") == "US":
            for rd in country.get("release_dates", []):
                cert = rd.get("certification", "").strip()
                if cert:
                    return cert
    return None

In [42]:
def parse_movie(data):
    """Extract all fields from a full movie detail response."""

    # Basic info 
    title    = data.get("title")
    year     = (data.get("release_date") or "")[:4] or None
    overview = data.get("overview") or None

    # Certificate
    certificate = parse_certificate(data.get("release_dates", {}))

    # Duration
    runtime  = data.get("runtime")
    duration = f"{runtime} min" if runtime else None

    # Genre
    genre = ", ".join(g["name"] for g in data.get("genres", []))

    # Ratings
    tmdb_rating = data.get("vote_average")
    votes       = data.get("vote_count")

    # Director & Stars from credits
    credits   = data.get("credits", {})
    crew      = credits.get("crew", [])
    cast      = credits.get("cast", [])

    director  = next((p["name"] for p in crew if p.get("job") == "Director"), None)
    stars     = [p["name"] for p in cast[:4]]   # top 4 billed actors

    # Revenue (box office)
    revenue = data.get("revenue")
    grossed = f"${revenue:,}" if revenue else None

    # Oringinal Language
    original_language = data.get("original_language")

    # Popularity
    popularity        = data.get("popularity")

    # TMDB & IMDb ID
    tmdb_id           = data.get("id")
    imdb_id           = data.get("imdb_id")

    return {
        "Movie Name":          title,
        "Year":                year,
        "Certificate":         certificate,
        "Duration":            duration,
        "Genre":               genre,
        "TMDB Rating":         tmdb_rating,
        "Votes":               votes,
        "Director":            director,
        "Stars":               stars,
        "Grossed in $":        grossed,
        "Plot":                overview,
        "Original Language":   original_language,
        "Popularity":          popularity,
        "TMDB ID":             tmdb_id,
        "IMDb ID":             imdb_id,
    }

In [43]:
# Main
if __name__ == "__main__":

    print(f"Fetching top-rated movie list")

    # Step 1: collect all movie IDs from top-rated pages
    all_ids = []
    for page in range(1, PAGES + 1):
        try:
            movies = get_top_rated_movies(page)
            all_ids.extend(m["id"] for m in movies)
            print(f"  Page {page}/{PAGES} — {len(all_ids)} movies so far")
        except Exception as e:
            print(f"  Page {page} failed: {e}")
        time.sleep(0.25)   # stay well under TMDB's 40 req/10s rate limit

    # Deduplicate
    all_ids = list(dict.fromkeys(all_ids))
    print(f"\nTotal unique movies: {len(all_ids)}")
    print("Fetching full details for each movie...\n")

    # Step 2: fetch full details for each movie
    movie_data = []
    for idx, movie_id in enumerate(all_ids):
        try:
            data    = get_movie_details(movie_id)
            details = parse_movie(data)
            movie_data.append(details)
            print(f"[{idx+1}/{len(all_ids)}] ✓ {details['Movie Name']} ({details['Year']})")
        except Exception as e:
            print(f"[{idx+1}/{len(all_ids)}] ✗ ID {movie_id}: {e}")
        time.sleep(0.26)   # ~3.8 req/s — safely within rate limit

    # Step 3: build DataFrame and save
    df = pd.DataFrame(movie_data)

    print(f"Done! Scraped {len(df)} movies.")
    print(df[["Movie Name", "Year", "TMDB Rating", "Director"]].head(10).to_string())

    df.to_parquet("tmdb_movies_raw.parquet")

Fetching top-rated movie list
  Page 1/500 — 20 movies so far
  Page 2/500 — 40 movies so far
  Page 3/500 — 60 movies so far
  Page 4/500 — 80 movies so far
  Page 5/500 — 100 movies so far
  Page 6/500 — 120 movies so far
  Page 7/500 — 140 movies so far
  Page 8/500 — 160 movies so far
  Page 9/500 — 180 movies so far
  Page 10/500 — 200 movies so far
  Page 11/500 — 220 movies so far
  Page 12/500 — 240 movies so far
  Page 13/500 — 260 movies so far
  Page 14/500 — 280 movies so far
  Page 15/500 — 300 movies so far
  Page 16/500 — 320 movies so far
  Page 17/500 — 340 movies so far
  Page 18/500 — 360 movies so far
  Page 19/500 — 380 movies so far
  Page 20/500 — 400 movies so far
  Page 21/500 — 420 movies so far
  Page 22/500 — 440 movies so far
  Page 23/500 — 460 movies so far
  Page 24/500 — 480 movies so far
  Page 25/500 — 500 movies so far
  Page 26/500 — 520 movies so far
  Page 27/500 — 540 movies so far
  Page 28/500 — 560 movies so far
  Page 29/500 — 580 movies so f

Kode melakukan iterasi melalui ratusan halaman API, mengumpulkan ID film, kemudian mengambil detail lengkap untuk setiap film tersebut.

Setelah data dikumpulkan ke dalam DataFrame Pandas, data tersebut dikonversi ke format file Parquet. Format Parquet merupakan format penyimpanan kolom yang efisien untuk pemrosesan Big Data dalam hal kompresi dan kecepatan akses dibandingkan format CSV tradisional, format ini juga memungkinkan proses pembacaan data yang lebih cepat pada sistem komputasi terdistribusi seperti Apache Spark.